# 📰 GDELT Chunked Download + Consolidation

## Strategy
Instead of fetching 3+ years in one long session (which risks Colab timeout),
this notebook breaks the date range into **manageable chunks** that you run
in separate Colab sessions.

## How to use
1. **Cell 2** — Set your date range chunk (e.g. Jan 2015 → Dec 2016)
2. Run **Cells 1–5** to fetch and download that chunk
3. Next session → change dates to the next chunk (Jan 2017 → Dec 2018)
4. Repeat until all chunks are downloaded
5. Upload all chunk files → run **Cell 6 (Consolidation)** to merge into one master file

## Suggested date chunks (3 months each = ~12 min per session)
```
Chunk 1 : 01-Jan-2015 → 31-Dec-2016
Chunk 2 : 01-Jan-2017 → 31-Dec-2018
Chunk 3 : 01-Jan-2019 → 31-Dec-2020
Chunk 4 : 01-Jan-2021 → 31-Dec-2022
Chunk 5 : 01-Jan-2023 → 30-Mar-2026  ← already downloaded
```

Each chunk file is named automatically: `gdelt_chunk_YYYYMMDD_YYYYMMDD.csv`

---
**Runtime → Run All** for each chunk session

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install requests pandas -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ═══════════════════════════════════════════════════════════════════════
# ▶▶  CHANGE THESE DATES FOR EACH SESSION  ◀◀
# ═══════════════════════════════════════════════════════════════════════
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import os
from google.colab import files

START_DATE = datetime(2015, 1, 1)    # ← CHANGE THIS
END_DATE   = datetime(2016, 12, 31)  # ← CHANGE THIS

# ── GDELT settings ────────────────────────────────────────────────────────
GDELT_BASE   = 'https://api.gdeltproject.org/api/v2/doc/doc'
MAX_RECORDS  = 250
CHUNK_DAYS   = 90       # 90-day chunks per API call
SLEEP_CHUNK  = 8.0      # 8s between chunks
SLEEP_TOPIC  = 10.0     # 10s between topics
SLEEP_429    = 60.0     # 60s wait on rate limit

# ── Output file name — auto-generated from date range ─────────────────────
OUTPUT_FILE = f"gdelt_chunk_{START_DATE.strftime('%Y%m%d')}_{END_DATE.strftime('%Y%m%d')}.csv"

# ── Top 10 high-signal topics ─────────────────────────────────────────────
TOPICS = {
    'NIFTY_50':            'Nifty India stock market',
    'RBI_REPO_RATE':       'Reserve Bank India repo rate policy',
    'USD_INR':             'dollar rupee exchange rate India currency',
    'CRUDE_OIL':           'Brent crude oil price India energy',
    'US_FED':              'Federal Reserve interest rate policy meeting',
    'FII_DII_FLOWS':       'foreign institutional investor India inflows',
    'GLOBAL_MARKETS':      'global stock markets Wall Street rally',
    'INDIA_CPI_INFLATION': 'India inflation consumer price index',
    'RELIANCE_INDUSTRIES': 'Reliance Industries stock market India',
    'HDFC_BANK':           'HDFC Bank India quarterly results',
}

print('=' * 60)
print('  GDELT CHUNKED DOWNLOAD')
print('=' * 60)
print(f'  Date range  : {START_DATE.date()} → {END_DATE.date()}')
print(f'  Topics      : {len(TOPICS)}')
print(f'  Output file : {OUTPUT_FILE}')
days    = (END_DATE - START_DATE).days
chunks  = (days // CHUNK_DAYS) + 1
est_min = (len(TOPICS) * chunks * (SLEEP_CHUNK + 2)) / 60
print(f'  Est. time   : ~{est_min:.0f}–{est_min*1.5:.0f} minutes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Test GDELT Connectivity
# ─────────────────────────────────────────────────────────────────────────
print('Testing GDELT connectivity...')
print('  Waiting 8s before first request...')
time.sleep(8)

test_params = {
    'query':         'Nifty India stock market sourcelang:english',
    'mode':          'artlist',
    'maxrecords':    5,
    'startdatetime': START_DATE.strftime('%Y%m%d') + '000000',
    'enddatetime':   (START_DATE + timedelta(days=30)).strftime('%Y%m%d') + '000000',
    'format':        'json',
    'sort':          'DateDesc',
}

def try_request(params, wait=0):
    if wait > 0:
        print(f'  Waiting {wait}s...')
        time.sleep(wait)
    return requests.get(GDELT_BASE, params=params, timeout=30)

try:
    resp = try_request(test_params)
    print(f'  Status : {resp.status_code}  ({len(resp.text)} chars)')

    if resp.status_code == 429:
        resp = try_request(test_params, wait=60)
        print(f'  Retry  : {resp.status_code}')

    if resp.status_code == 200 and len(resp.text.strip()) > 10:
        try:
            data = resp.json()
            arts = data.get('articles', [])
            print(f'  Articles found : {len(arts)}')
            if arts:
                print(f'  Sample title   : {arts[0].get("title","")[:70]}')
            print()
            print('  ✅ GDELT working — proceed to Cell 4')
        except Exception as e:
            print(f'  ⚠️  JSON error: {e}')
            print(f'  Raw: {resp.text[:100]}')
    elif resp.status_code == 429:
        print('  ❌ Still rate limited — try: Runtime → Disconnect and delete runtime')
    else:
        print(f'  ❌ Unexpected response: {resp.text[:100]}')
except Exception as e:
    print(f'  ❌ Connection error: {e}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Fetch All Topics for This Date Range
# ─────────────────────────────────────────────────────────────────────────

def fetch_topic(topic_name, query, start_dt, end_dt):
    all_articles  = []
    current_start = start_dt

    while current_start < end_dt:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS), end_dt)

        params = {
            'query':         f'{query} sourcelang:english',
            'mode':          'artlist',
            'maxrecords':    MAX_RECORDS,
            'startdatetime': current_start.strftime('%Y%m%d%H%M%S'),
            'enddatetime':   current_end.strftime('%Y%m%d%H%M%S'),
            'format':        'json',
            'sort':          'DateDesc',
        }

        retry = 0
        while retry < 3:
            try:
                resp = requests.get(GDELT_BASE, params=params, timeout=30)

                if resp.status_code == 429:
                    print(f'    ⚠️  Rate limit — waiting {SLEEP_429:.0f}s  (retry {retry+1}/3)')
                    time.sleep(SLEEP_429)
                    retry += 1
                    continue

                if len(resp.text.strip()) == 0:
                    print(f'    ⚠️  Empty response — retry {retry+1}/3')
                    time.sleep(6)
                    retry += 1
                    continue

                if resp.status_code != 200:
                    print(f'    ⚠️  HTTP {resp.status_code} — skipping chunk')
                    break

                try:
                    data     = resp.json()
                except Exception:
                    print(f'    ⚠️  JSON error — retry {retry+1}/3  ({resp.text[:60]})')
                    time.sleep(6)
                    retry += 1
                    continue

                articles = data.get('articles', [])
                for a in articles:
                    all_articles.append({
                        'topic':        topic_name,
                        'published_at': a.get('seendate', ''),
                        'source':       a.get('domain', ''),
                        'title':        a.get('title', ''),
                        'url':          a.get('url', ''),
                        'language':     a.get('language', ''),
                        'tone':         a.get('tone', None),
                        'country':      a.get('sourcecountry', ''),
                    })

                label = f"{current_start.strftime('%d-%b-%Y')}–{current_end.strftime('%d-%b-%Y')}"
                print(f'    {label}: {len(articles):>3} articles  (total: {len(all_articles)})')
                break

            except requests.exceptions.Timeout:
                print(f'    ⚠️  Timeout — retry {retry+1}/3')
                retry += 1
                time.sleep(5)
            except Exception as e:
                print(f'    ⚠️  Error: {str(e)[:60]} — retry {retry+1}/3')
                retry += 1
                time.sleep(5)

        current_start = current_end
        time.sleep(SLEEP_CHUNK)

    return all_articles


# ── Run all topics ─────────────────────────────────────────────────────────
all_articles  = []
topic_summary = {}
failed_topics = []

print('=' * 60)
print(f'  FETCHING {len(TOPICS)} TOPICS')
print(f'  {START_DATE.date()} → {END_DATE.date()}')
print('=' * 60)

for idx, (topic_name, query) in enumerate(TOPICS.items(), 1):
    print(f'\n[{idx:>2}/{len(TOPICS)}] {topic_name}')
    print(f'  Query: {query}')

    articles = fetch_topic(topic_name, query, START_DATE, END_DATE)
    topic_summary[topic_name] = len(articles)

    if articles:
        all_articles.extend(articles)
        print(f'  ✓ {len(articles)} articles fetched')
    else:
        failed_topics.append(topic_name)
        print(f'  ⚠️  0 articles')

    time.sleep(SLEEP_TOPIC)

print()
print('=' * 60)
print('  FETCH COMPLETE')
print('=' * 60)
print(f'  Total articles : {len(all_articles)}')
print(f'  Topics success : {len(TOPICS) - len(failed_topics)} / {len(TOPICS)}')
if failed_topics:
    print(f'  Failed topics  : {failed_topics}')
print()
print('  Articles per topic:')
for t, c in sorted(topic_summary.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * min(c // 20, 25)
    print(f'  {t:<30} : {c:>4}  {bar}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Save and Download This Chunk
# ─────────────────────────────────────────────────────────────────────────

if not all_articles:
    print('⚠️  No articles to save.')
else:
    chunk_df = pd.DataFrame(all_articles)

    # Parse dates
    chunk_df['published_at'] = pd.to_datetime(
        chunk_df['published_at'], format='%Y%m%dT%H%M%SZ', errors='coerce'
    )
    chunk_df['date'] = chunk_df['published_at'].dt.date

    # Remove duplicates
    chunk_df = chunk_df.drop_duplicates(subset=['url', 'topic'])
    chunk_df = chunk_df.sort_values('published_at', ascending=False)
    chunk_df['tone'] = pd.to_numeric(chunk_df['tone'], errors='coerce')

    # Save
    chunk_df.to_csv(OUTPUT_FILE, index=False)
    size = os.path.getsize(OUTPUT_FILE)

    print('=' * 60)
    print(f'  CHUNK SAVED: {OUTPUT_FILE}')
    print('=' * 60)
    print(f'  Rows          : {len(chunk_df)}')
    print(f'  Date range    : {chunk_df["date"].min()} → {chunk_df["date"].max()}')
    print(f'  Unique dates  : {chunk_df["date"].nunique()}')
    print(f'  File size     : {size/1024:.1f} KB')
    print()
    print('⬇️  Downloading...')
    files.download(OUTPUT_FILE)
    print(f'  ✓ {OUTPUT_FILE} downloaded')
    print()
    print('✅ Chunk complete!')
    print()
    print('  Next steps:')
    print('  1. Note down this filename — you will need it for consolidation')
    print('  2. Start a new Colab session')
    print('  3. Change START_DATE and END_DATE in Cell 2 to the next chunk')
    print('  4. Run all cells again')
    print('  5. Once ALL chunks downloaded → run Cell 6 to consolidate')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# CELL 6 — CONSOLIDATION (Run this ONLY after all chunks are downloaded)
#
# Upload all chunk files at once → this cell merges them into one
# master file: step5a_raw_news_all_topics.csv
# ═════════════════════════════════════════════════════════════════════════
print('=' * 60)
print('  CONSOLIDATION — Upload ALL chunk files')
print('=' * 60)
print()
print('Upload all your chunk CSV files now (select multiple files):')
print('e.g. gdelt_chunk_20150101_20161231.csv')
print('     gdelt_chunk_20170101_20181231.csv')
print('     gdelt_chunk_20190101_20201231.csv')
print('     ... etc')
print()

uploaded = files.upload()

print(f'\n  Files uploaded: {len(uploaded)}')
for fname in uploaded.keys():
    print(f'    ✓ {fname}')

# Load and concatenate all chunks
dfs = []
for fname in uploaded.keys():
    try:
        df = pd.read_csv(fname)
        dfs.append(df)
        print(f'  Loaded {fname}: {len(df)} rows')
    except Exception as e:
        print(f'  ⚠️  Error loading {fname}: {e}')

if not dfs:
    print('❌ No files loaded — check your uploads')
else:
    master_df = pd.concat(dfs, ignore_index=True)

    # Parse dates
    master_df['published_at'] = pd.to_datetime(
        master_df['published_at'], errors='coerce'
    )
    master_df['date'] = master_df['published_at'].dt.date

    # Remove duplicates across chunks
    before = len(master_df)
    master_df = master_df.drop_duplicates(subset=['url', 'topic'])
    after  = len(master_df)
    print(f'\n  Duplicates removed : {before - after}')

    # Sort by date ascending
    master_df = master_df.sort_values('published_at').reset_index(drop=True)
    master_df['tone'] = pd.to_numeric(master_df['tone'], errors='coerce')

    # Save master file
    MASTER_FILE = 'step5a_raw_news_all_topics.csv'
    master_df.to_csv(MASTER_FILE, index=False)
    size = os.path.getsize(MASTER_FILE)

    print()
    print('=' * 60)
    print('  MASTER FILE CREATED')
    print('=' * 60)
    print(f'  File          : {MASTER_FILE}')
    print(f'  Total rows    : {len(master_df)}')
    print(f'  Date range    : {master_df["date"].min()} → {master_df["date"].max()}')
    print(f'  Unique dates  : {master_df["date"].nunique()}')
    print(f'  Unique topics : {master_df["topic"].nunique()}')
    print(f'  File size     : {size/1024/1024:.1f} MB')
    print()
    print('  Articles per topic:')
    topic_counts = master_df['topic'].value_counts()
    for topic, count in topic_counts.items():
        bar = '█' * min(count // 100, 25)
        print(f'    {topic:<30} : {count:>5}  {bar}')
    print()
    print('  Coverage by chunk:')
    master_df['year_month'] = master_df['published_at'].dt.to_period('Y')
    yearly = master_df.groupby('year_month').size().sort_index()
    for period, count in yearly.items():
        bar = '█' * min(count // 100, 25)
        print(f'    {period}: {count:>5} articles  {bar}')
    print()
    print('⬇️  Downloading master file...')
    files.download(MASTER_FILE)
    print(f'  ✓ {MASTER_FILE} downloaded')
    print()
    print('✅ Consolidation complete!')
    print('  Next → Run Step 6 (GPT-4o) with this master file')